In [3]:
from pyspark.sql import SparkSession, functions as f 
from pyspark.sql.types import StructField, StructType, IntegerType, StringType, DecimalType, BooleanType
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("CarPrice").getOrCreate()

file_path = "/Users/tejamarneni/Desktop/Datasets/used_car_price_prediction_1M.csv"

dataset_schema = StructType([
                    StructField("Brand",StringType(),True)
                   ,StructField("Model",StringType(),True)
                   ,StructField("Year",IntegerType(),True)
                   ,StructField("Mileage_kmpl", DecimalType(10,2))
                   ,StructField("Engine_CC", DecimalType(10,2))
                   ,StructField("Horsepower", DecimalType(6,2))
                   ,StructField("Fuel_Type",StringType(),True)
                   ,StructField("Transmission",StringType(),True)
                   ,StructField("Owner_Type",StringType(),True)
                   ,StructField("Color",StringType(),True)
                   ,StructField("City",StringType(),True)
                   ,StructField("Kms_Driven",DecimalType(10,2),True)
                   ,StructField("Insurance_Valid",IntegerType(),True)
                   ,StructField("Service_History",IntegerType(),True)
                   ,StructField("Accidents",IntegerType(),True)
                   ,StructField("Tax_Paid",IntegerType(),True)
                   ,StructField("Number_of_Doors",IntegerType(),True)
                   ,StructField("Seats",IntegerType(),True)
                   ,StructField("Registration_Age",IntegerType(),True)
                   ,StructField("Price",DecimalType(10,2),True)
                ])

df = spark.read.format("csv") \
    .option("header", "true") \
    .option("ignoreLeadingWhiteSpace", "true") \
    .option("ignoreTrailingWhiteSpace", "true") \
    .schema(dataset_schema) \
    .load(file_path)

df.show(5)

+--------+------+----+------------+---------+----------+---------+------------+----------+-----+-------+----------+---------------+---------------+---------+--------+---------------+-----+----------------+----------+
|   Brand| Model|Year|Mileage_kmpl|Engine_CC|Horsepower|Fuel_Type|Transmission|Owner_Type|Color|   City|Kms_Driven|Insurance_Valid|Service_History|Accidents|Tax_Paid|Number_of_Doors|Seats|Registration_Age|     Price|
+--------+------+----+------------+---------+----------+---------+------------+----------+-----+-------+----------+---------------+---------------+---------+--------+---------------+-----+----------------+----------+
|  Toyota| Camry|2001|       19.62|  1396.56|      NULL|   Hybrid|   Automatic|     First| Grey|Chennai| 500928.00|              1|              1|        0|       1|              4|    5|              24| 353189.60|
| Hyundai|   i20|2012|       19.48|  1130.77|    137.02|   Petrol|        NULL|     First| Grey| Mumbai| 211510.00|              0| 

In [4]:
df_avg_kms_Model = df.groupBy(["Brand","Model"]).agg(f.round(f.avg(f.col("Kms_Driven")),2).alias("Avg_Kms")).orderBy(f.desc("Avg_Kms"))
df_avg_kms_Model.show(50)

+----------+---------+---------+
|     Brand|    Model|  Avg_Kms|
+----------+---------+---------+
|Volkswagen|   Virtus|205908.36|
|  Mercedes|  E-Class|205044.63|
|      Tata|  Harrier|204797.63|
|       BMW|       X3|204690.29|
|       Kia|   Seltos|204156.33|
|      Tata|    Nexon|204121.44|
|     Honda|    Civic|203887.16|
|       BMW|       X1|203780.78|
|    Nissan|    Kicks|203767.17|
|       Kia|    Sonet|203734.31|
|    Toyota|    Yaris|203675.28|
|  Mahindra|  Scorpio|203574.04|
|      Ford| EcoSport|203456.95|
|     Skoda|   Kushaq|203433.76|
|  Mercedes|      GLA|203313.06|
|      Audi|       Q5|203229.92|
|  Mahindra|   XUV700|203063.42|
|      Tata|    Punch|203043.45|
|    Nissan|    Sunny|202995.36|
|    Nissan|  Magnite|202841.67|
|    Toyota|    Camry|202702.60|
|  Mahindra|   Bolero|202589.30|
|      Ford|Endeavour|202485.51|
|     Skoda|   Slavia|202484.72|
|Volkswagen|   Taigun|202443.48|
|   Hyundai|    Creta|202401.93|
|      Audi|       A4|202386.07|
|       Ki

In [5]:
df.select("Insurance_Valid").distinct().show()

+---------------+
|Insurance_Valid|
+---------------+
|              1|
|              0|
+---------------+



In [6]:
df_avg_price = df.groupBy("Brand").agg(f.round(f.avg(f.col("Price")),2).alias("Avg_Price")).orderBy(f.desc("Avg_Price"))
df_avg_price.show()

+----------+----------+
|     Brand| Avg_Price|
+----------+----------+
|  Mercedes|2515450.91|
|       BMW|2191307.60|
|      Audi|2082615.80|
|Volkswagen| 792197.70|
|     Skoda| 745236.70|
|       Kia| 744362.57|
|     Honda| 650912.51|
|    Toyota| 647854.52|
|   Hyundai| 605701.20|
|  Mahindra| 558638.93|
|    Nissan| 558469.73|
|      Ford| 555292.58|
|      Tata| 513824.73|
+----------+----------+



In [7]:
df_avg_price_Model = df.groupBy(["Brand","Model"]).agg(f.round(f.avg(f.col("Price")),2).alias("Avg_Price")).orderBy(f.desc("Avg_Price"))
df_avg_price_Model.show(50)

+----------+---------+----------+
|     Brand|    Model| Avg_Price|
+----------+---------+----------+
|  Mercedes|  E-Class|2525026.78|
|  Mercedes|      GLA|2511275.83|
|  Mercedes|  C-Class|2509992.54|
|       BMW|       X1|2201460.52|
|       BMW|       X3|2186940.04|
|       BMW| 3 Series|2185570.04|
|      Audi|       Q3|2098918.42|
|      Audi|       A4|2079561.60|
|      Audi|       Q5|2069263.20|
|Volkswagen|     Polo| 797574.27|
|Volkswagen|   Taigun| 796171.90|
|Volkswagen|   Virtus| 782893.42|
|       Kia|   Carens| 752540.78|
|     Skoda|  Octavia| 747200.00|
|     Skoda|   Kushaq| 744834.24|
|     Skoda|   Slavia| 743650.14|
|       Kia|    Sonet| 742236.13|
|       Kia|   Seltos| 738283.52|
|     Honda|    Amaze| 655655.10|
|     Honda|    Civic| 650258.03|
|    Toyota|  Corolla| 649654.94|
|    Toyota|    Yaris| 647133.00|
|    Toyota|    Camry| 646794.28|
|     Honda|     City| 646768.06|
|   Hyundai|    Creta| 608342.96|
|   Hyundai|    Verna| 606034.78|
|   Hyundai|  

In [8]:
df_med_price = df.groupBy("Brand").agg(f.round(f.median(f.col("Price")),2).alias("Median_Price")).orderBy(f.desc("Median_Price"))
df_med_price.show()

+----------+------------+
|     Brand|Median_Price|
+----------+------------+
|  Mercedes|  2402907.53|
|       BMW|  2099703.19|
|      Audi|  1996373.79|
|Volkswagen|   745318.53|
|     Skoda|    700742.9|
|       Kia|   697309.66|
|     Honda|    601088.2|
|    Toyota|   598276.74|
|   Hyundai|   554232.81|
|  Mahindra|   500086.57|
|    Nissan|   499470.63|
|      Ford|    499148.7|
|      Tata|   450583.04|
+----------+------------+



In [8]:
df_Brand = df.groupBy("Brand").agg(f.count("Brand").alias("Brand_Count")).orderBy(f.desc("Brand_Count"))
df_Brand.show(30)

+----------+-----------+
|     Brand|Brand_Count|
+----------+-----------+
|       BMW|      77561|
|     Skoda|      77551|
|   Hyundai|      77517|
|      Ford|      77439|
|    Toyota|      77397|
|     Honda|      77312|
|      Audi|      77306|
|  Mercedes|      77305|
|       Kia|      77293|
|Volkswagen|      77185|
|  Mahindra|      77093|
|      Tata|      77075|
|    Nissan|      76966|
+----------+-----------+



In [9]:
df_Brand = df.withColumn("Brand",f.trim(f.col("Brand"))) \
             .groupBy("Brand").agg(f.count("Brand").alias("Brand_Count")).orderBy(f.desc("Brand_Count"))
df_Brand.show(30)

+----------+-----------+
|     Brand|Brand_Count|
+----------+-----------+
|       BMW|      77561|
|     Skoda|      77551|
|   Hyundai|      77517|
|      Ford|      77439|
|    Toyota|      77397|
|     Honda|      77312|
|      Audi|      77306|
|  Mercedes|      77305|
|       Kia|      77293|
|Volkswagen|      77185|
|  Mahindra|      77093|
|      Tata|      77075|
|    Nissan|      76966|
+----------+-----------+

